In [1]:
import os
import gc
import random
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor


def set_seed(seed=2021):
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed()




In [2]:
train_usecols = ["R", "C", "time_step", "u_in", "u_out", "pressure", "breath_id"]
dtypes = {
    "R": "int8",
    "C": "int8",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "int8",
    "pressure": "float32",
    "breath_id": "int32",
}
df_train = pd.read_csv(
    "../input/ventilator-pressure-prediction/train.csv",
    usecols=train_usecols,
    dtype=dtypes,
    low_memory=False,
    memory_map=True,
)




In [3]:
def add_engineered_features(df):
    # keep original ordering per breath
    df = df.sort_values(["breath_id", "time_step"]).reset_index(drop=True)

    # basic differences
    dt = df.groupby("breath_id")["time_step"].diff().fillna(0).astype(np.float32)

    flow = df["u_in"].astype(np.float32) - df["u_out"].astype(np.float32)

    cum_vol = (dt * flow).groupby(df["breath_id"]).cumsum().astype(np.float32)

    # original interaction terms
    r_c = (
        (df["R"].astype(np.float32) * df["C"].astype(np.float32))
        .to_numpy()
        .reshape(-1, 1)
    )
    uin_r = (
        (df["u_in"].astype(np.float32) * df["R"].astype(np.float32))
        .to_numpy()
        .reshape(-1, 1)
    )
    uin_c = (
        (df["u_in"].astype(np.float32) * df["C"].astype(np.float32))
        .to_numpy()
        .reshape(-1, 1)
    )

    # new simple polynomial features
    time_sq = (df["time_step"].astype(np.float32) ** 2).to_numpy().reshape(-1, 1)
    uin_sq = (df["u_in"].astype(np.float32) ** 2).to_numpy().reshape(-1, 1)

    base = df[["R", "C", "time_step", "u_in", "u_out"]].to_numpy(
        dtype=np.float32, copy=False
    )
    extra = np.column_stack(
        (
            dt.values,
            cum_vol.values,
            r_c,
            uin_r,
            uin_c,
            time_sq,
            uin_sq,
        )
    )
    return np.hstack((base, extra))


def baseline_prediction(train_df):
    test_usecols = ["R", "C", "time_step", "u_in", "u_out", "breath_id"]
    test_dtypes = {
        "R": "int8",
        "C": "int8",
        "time_step": "float32",
        "u_in": "float32",
        "u_out": "int8",
        "breath_id": "int32",
    }
    test_df = pd.read_csv(
        "../input/ventilator-pressure-prediction/test.csv",
        usecols=test_usecols,
        dtype=test_dtypes,
        low_memory=False,
        memory_map=True,
    )

    X_train = add_engineered_features(train_df)
    y_train = train_df["pressure"].astype(np.float32).values
    X_test = add_engineered_features(test_df)

    model = HistGradientBoostingRegressor(
        max_iter=2000,  # more boosting rounds
        learning_rate=0.03,  # finer step size
        max_depth=8,  # deeper trees
        l2_regularization=0.1,  # mild regularisation
        max_bins=255,
        random_state=2021,
        early_stopping=True,  # stop if no improvement
        validation_fraction=0.1,  # hold‑out for early‑stopping
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    # clip to realistic pressure range observed in training
    preds = np.clip(preds, y_train.min(), y_train.max())

    submission = pd.read_csv(
        "../input/ventilator-pressure-prediction/sample_submission.csv"
    )
    submission["pressure"] = preds
    submission.to_csv("submission.csv", index=False)

    # clean up
    del test_df, X_train, y_train, X_test, model, preds, submission
    gc.collect()




In [4]:
baseline_prediction(df_train)